# Python 6: files, paths and formats

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-06-files-and-formats.ipynb)

Most pipelines start and end with files: CSV exports, JSON from an API, compressed logs and folders of partitioned output. This chapter covers how to build paths without string gluing, how to open files so they are always closed, how to read and write CSV, JSON and JSON Lines, how to handle large and compressed files, and how to write a file so a crash never leaves a half-written one behind.

Every example writes into one temporary folder. The last cell deletes it, so the notebook leaves no files behind. The rules below are stated before the code that shows them, and each `assert` is the expected result.

## Setup: a temporary folder for the examples

`tempfile.mkdtemp()` creates a new empty folder that the operating system keeps out of the way. The examples write into it, and the last cell removes it.

In [ ]:
import csv
import gzip
import io
import json
import os
import shutil
import tempfile
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix="py-files-"))
landing = workdir / "landing" / "orders"
landing.mkdir(parents=True)
assert landing.is_dir()

## Paths with `pathlib`

`pathlib.Path` joins paths with `/`, which works on every operating system. A `Path` also exposes the parts of a name without string slicing:

- `name` is the last part, `stem` is the name without its suffix, and `suffix` is the extension, including the dot.
- `parent` is the folder that contains the path.
- `with_suffix(".json")` returns a new path with a different extension.
- `glob("pattern")` finds matching names in one folder. `rglob("pattern")` searches subfolders as well.

### Where you'll use it

- **Discovering input files:** `glob` the day's landing folder, then process the matches in sorted order, so reruns see the same sequence.
- **Building partition paths:** `base / "dt=2024-01-15" / "part-0000.parquet"` in a writer.

In [ ]:
file_path = landing / "orders_2024-01-15.csv"
assert file_path.name == "orders_2024-01-15.csv"
assert file_path.stem == "orders_2024-01-15" and file_path.suffix == ".csv"
assert file_path.parent == landing
assert file_path.with_suffix(".json").name == "orders_2024-01-15.json"
assert landing.parts[-2:] == ("landing", "orders")
assert not file_path.exists()             # a path can name a file that does not exist yet

for day in ["2024-01-16", "2024-01-15"]:
    (landing / f"orders_{day}.csv").write_text("order_id\n1\n", encoding="utf-8")

# glob returns matches in arbitrary order, so sort them for a repeatable sequence.
names = sorted(path.name for path in landing.glob("orders_*.csv"))
assert names == ["orders_2024-01-15.csv", "orders_2024-01-16.csv"]

## Context managers: `with` closes what it opened

A **context manager** is an object that sets something up when a `with` block starts and cleans it up when the block ends. The cleanup runs **even if an error is raised inside the block**. Opening a file is the classic case: `with open(...) as f:` opens the file and guarantees it is closed.

Without `with`, the file stays open until Python garbage-collects it, and an error in the middle can leave data unflushed. Always use `with` for files, connections and locks.

### Where you'll use it

- **Connections and transactions:** `with db.connect() as conn:` closes the connection even when a load fails halfway.
- **Locks and temporary state:** a `with lock:` block releases the lock on error, so the next worker is not blocked forever.

In [ ]:
note = workdir / "note.txt"
with note.open("w", encoding="utf-8") as handle:
    handle.write("hello")
    assert not handle.closed              # the file is open inside the block

assert handle.closed                      # closed when the block ended

try:
    with note.open(encoding="utf-8") as handle:
        raise RuntimeError("boom")        # an error inside the block...
except RuntimeError:
    pass
assert handle.closed                      # ...and the file was still closed

## Reading and writing text

Text files are bytes that a codec turns into characters. Always pass `encoding="utf-8"` explicitly. Without it, Python uses the machine's default encoding, so the same file can read differently on another computer.

For CSV, open the file with `newline=""`. The `csv` module then handles the line endings itself, and a quoted field that contains a newline survives.

### Where you'll use it

- **Ingesting vendor files:** a file with a UTF-8 byte-order mark, or with Windows line endings, that breaks a naive reader.
- **Writing logs and reports:** an explicit encoding, so the file reads the same on every machine.

In [ ]:
text_path = landing / "notes.txt"
with text_path.open("w", encoding="utf-8") as handle:
    handle.write("first line\n")
    handle.write("café\n")

with text_path.open(encoding="utf-8") as handle:
    lines = handle.read().splitlines()          # splitlines drops the line endings

assert lines == ["first line", "café"]
assert text_path.read_bytes() == "first line\ncafé\n".encode("utf-8")   # é is two bytes

### Byte-order marks

Some Windows tools write a **byte-order mark** (BOM), the three bytes that start a UTF-8 file. Read as plain `utf-8`, the BOM becomes an invisible character `﻿` at the start of the first column name, so `"id"` no longer matches. The `utf-8-sig` codec removes it.

In [ ]:
bom_path = landing / "vendor.csv"
bom_path.write_bytes("﻿id,name\n1,ana\n".encode("utf-8"))

with bom_path.open(encoding="utf-8") as handle:
    assert handle.readline() == "﻿id,name\n"      # the BOM is still in the text

with bom_path.open(encoding="utf-8-sig") as handle:
    assert handle.readline() == "id,name\n"            # utf-8-sig strips it

## CSV with the `csv` module

`csv.DictReader` yields one dictionary per row, keyed by the header. `csv.DictWriter` writes dictionaries back out, in the order of `fieldnames`. Every value read from a CSV file is a **string**, so convert the types yourself.

Never build CSV by joining strings with commas. A value that contains a comma, a quote or a newline must be quoted, and the `csv` module does that for you.

### Where you'll use it

- **Daily exports:** CSV is still the most common handoff format from finance systems and SaaS tools.
- **Other delimiters:** European exports often use `;` and a decimal comma. Pass `delimiter=";"` and convert the number yourself.

In [ ]:
orders_path = landing / "orders.csv"
rows = [
    {"order_id": 1, "customer": "Ana, Ltd", "amount": 120.5},
    {"order_id": 2, "customer": "Ben", "amount": 80},
]
with orders_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["order_id", "customer", "amount"])
    writer.writeheader()
    writer.writerows(rows)

written = orders_path.read_text(encoding="utf-8").splitlines()
assert written[0] == "order_id,customer,amount"
assert written[1] == '1,"Ana, Ltd",120.5'           # the comma inside the value is quoted

with orders_path.open(newline="", encoding="utf-8") as handle:
    loaded = [
        {**row, "order_id": int(row["order_id"]), "amount": float(row["amount"])}
        for row in csv.DictReader(handle)
    ]
assert loaded[0]["customer"] == "Ana, Ltd"
assert loaded[1]["amount"] == 80.0

In [ ]:
# The same module reads an in-memory semicolon file, which is handy for tests.
euro = "order_id;amount\n7;3,50\n"
euro_rows = list(csv.DictReader(io.StringIO(euro), delimiter=";"))
assert euro_rows == [{"order_id": "7", "amount": "3,50"}]
assert float(euro_rows[0]["amount"].replace(",", ".")) == 3.5

assert list(csv.reader(io.StringIO('a,"b,c",d\n')))[0] == ["a", "b,c", "d"]

## JSON

`json.dumps` turns Python data into a JSON string, and `json.loads` turns it back. `json.dump` and `json.load` do the same with a file. JSON has only strings, numbers, booleans, `null`, arrays and objects. It has **no dates and no decimals**, so you convert them first. JSON object keys are always strings, even when Python had integer keys.

`sort_keys=True` writes the keys in a fixed order, which makes the output comparable between runs.

### Where you'll use it

- **API responses:** most REST APIs return JSON, which you parse, flatten and load.
- **Writing payloads:** a `default=` function converts dates and decimals when you serialise, instead of you converting them by hand in every place.

In [ ]:
event = {"id": 7, "tags": ["new", "mobile"]}
text = json.dumps(event, sort_keys=True)
assert text == '{"id": 7, "tags": ["new", "mobile"]}'
assert json.loads(text) == event

assert json.loads(json.dumps({1: "a"})) == {"1": "a"}       # the integer key became a string

try:
    json.dumps({"day": date(2024, 1, 15)})                   # dates are not JSON
except TypeError:
    pass
else:
    raise AssertionError("expected TypeError")

def to_json_value(value):
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)                                    # keep the exact digits
    raise TypeError(f"not JSON serialisable: {type(value).__name__}")

payload = {"day": date(2024, 1, 15), "total": Decimal("12.50")}
assert json.dumps(payload, default=to_json_value, sort_keys=True) == \
    '{"day": "2024-01-15", "total": "12.50"}'

In [ ]:
# JSON Lines: one JSON object per line. The standard format for logs and event streams.
events_path = landing / "events.jsonl"
with events_path.open("w", encoding="utf-8") as handle:
    for n in range(3):
        handle.write(json.dumps({"n": n}) + "\n")

with events_path.open(encoding="utf-8") as handle:
    assert [json.loads(line)["n"] for line in handle] == [0, 1, 2]

## Large files: read line by line

`read()` loads the whole file into memory. Iterating over the file object reads one line at a time, so memory stays flat even for a file far larger than RAM. Process each line as it arrives, and keep only the totals.

### Where you'll use it

- **Multi-gigabyte logs:** scanning a web server log for error codes without loading the whole file.
- **Streaming conversions:** converting a JSON Lines export to CSV, one line at a time.

In [ ]:
big_path = landing / "big.txt"
with big_path.open("w", encoding="utf-8") as handle:
    for n in range(100_000):
        handle.write(f"{n}\n")

total = 0
with big_path.open(encoding="utf-8") as handle:
    for line in handle:                  # one line in memory at a time
        total += int(line)
assert total == sum(range(100_000))

## Compressed files

`gzip.open` reads and writes compressed data with the same interface as `open`. Use `"rt"` to read text and `"wt"` to write text, and `"rb"` or `"wb"` for raw bytes. Compressed files are smaller on disk and cheaper to move, and most engines read them directly. Every gzip file begins with the two bytes `\x1f\x8b`, which is how tools recognise one.

### Where you'll use it

- **Data lakes:** most landing zones store `.gz` files, which engines read directly when you point them at the folder.
- **Shipping data:** a gzip file takes a fraction of the network time of the raw text.

In [ ]:
gz_path = landing / "events.jsonl.gz"
with gzip.open(gz_path, "wt", encoding="utf-8") as handle:
    handle.write('{"n": 1}\n{"n": 2}\n')

with gzip.open(gz_path, "rt", encoding="utf-8") as handle:
    values = [json.loads(line)["n"] for line in handle]
assert values == [1, 2]
assert gz_path.read_bytes()[:2] == b"\x1f\x8b"     # the gzip magic number

## Writing a file atomically

If a job crashes while it writes, a half-written file can look like a finished one. A **checkpoint** or a **published report** must never be read in that state.

The fix is to write to a temporary name in the same folder, flush it to disk, and then `replace` it into place. On one filesystem, `replace` is atomic: a reader sees the old file or the new file, never a mix.

### Where you'll use it

- **Checkpoints and manifests:** a job saves its state this way, so a crash never leaves a half-written state file for the next run to trust.
- **Published outputs:** a dashboard that reads a report must never see a partly written one.

In [ ]:
def write_atomic(path, text):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        handle.write(text)
        handle.flush()
        os.fsync(handle.fileno())        # force the bytes to disk before the rename
    tmp.replace(path)

target = landing / "status.json"
write_atomic(target, '{"ok": true}')
assert json.loads(target.read_text(encoding="utf-8")) == {"ok": True}
assert not (landing / "status.json.tmp").exists()

## Practice questions

1. Write `csv_files(folder)` that returns every `.csv` file under `folder`, recursively, sorted by file name. Use `pathlib`, not `os.walk`.
2. Write `count_gzip_lines(path)` that counts the lines of a `.gz` file without loading the file into memory.
3. Write `values_for_key(jsonl_path, key)` that returns one value per line of a JSON Lines file. Skip blank lines, because `json.loads("")` raises an error.

## Answers

### Question 1

`rglob` finds files in every subfolder. Its order is not guaranteed, so sort the result.

In [ ]:
tree = workdir / "tree"
(tree / "2024" / "01").mkdir(parents=True)
(tree / "b.csv").write_text("x\n", encoding="utf-8")
(tree / "2024" / "01" / "a.csv").write_text("x\n", encoding="utf-8")
(tree / "notes.txt").write_text("x\n", encoding="utf-8")

def csv_files(folder):
    return sorted(Path(folder).rglob("*.csv"), key=lambda path: path.name)

assert [path.name for path in csv_files(tree)] == ["a.csv", "b.csv"]

### Question 2

In [ ]:
def count_gzip_lines(path):
    with gzip.open(path, "rt", encoding="utf-8") as handle:
        return sum(1 for _ in handle)

assert count_gzip_lines(gz_path) == 2

### Question 3

In [ ]:
def values_for_key(jsonl_path, key):
    with Path(jsonl_path).open(encoding="utf-8") as handle:
        return [json.loads(line)[key] for line in handle if line.strip()]

blank_path = landing / "blank.jsonl"
blank_path.write_text('{"k": 1}\n\n{"k": 2}\n', encoding="utf-8")
assert values_for_key(blank_path, "k") == [1, 2]

## Clean up

Remove the temporary folder. Nothing after this cell uses it.

In [ ]:
shutil.rmtree(workdir)
assert not workdir.exists()

## Interview notes

- **Always use `with`, and always name the encoding.** Most "works on my machine" bugs are encoding, newline or BOM bugs.
- **Stream large files.** Iterate over the file object and do not call `read()` on a file that might be large.
- **Write atomically.** A temporary file plus a rename turns a crash mid-write into "no output", which a retry can fix.
- **Convert CSV values yourself.** Every field is a string. Check each conversion, because it is where bad rows are found.
- **Serialise dates and decimals explicitly.** JSON has no types for them, so decide the format once, in one function.
- **Compare the formats.** CSV has no types, JSON has no dates, JSON Lines suits streams, and Parquet suits analytics. Choose the format for how the data will be read later.